# Compare-Map Box Plots


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from IPython.display import display

COMPARE_OUTPUT_DIR_NAME = "compare-map-150"


def find_compare_output_dir() -> Path:
    candidates: list[Path] = []

    if "__file__" in globals():
        candidates.append(Path(__file__).resolve().parent)

    vscode_notebook = globals().get("__vsc_ipynb_file__")
    if vscode_notebook:
        candidates.append(Path(vscode_notebook).expanduser().resolve().parent)

    cwd = Path.cwd().resolve()
    candidates.extend(
        [
            cwd,
            cwd / "nas" / COMPARE_OUTPUT_DIR_NAME,
            cwd.parent / COMPARE_OUTPUT_DIR_NAME,
        ]
    )

    for candidate in candidates:
        if candidate.exists() and any(candidate.glob("*/metrics.jsonl")):
            return candidate

    raise FileNotFoundError(
        "Could not locate compare output directory. "
        f"Set BASE_OUTPUT_DIR manually to the folder containing metrics.jsonl subdirectories. Tried: {candidates}"
    )


BASE_OUTPUT_DIR = find_compare_output_dir()
print(f"Using compare output directory: {BASE_OUTPUT_DIR}")


def latest_run_dir(base_dir: Path = BASE_OUTPUT_DIR) -> Path:
    candidates = sorted(
        [path for path in base_dir.glob("*/metrics.jsonl")],
        key=lambda p: p.stat().st_mtime,
    )
    if not candidates:
        raise FileNotFoundError(
            f"No metrics.jsonl files found under {base_dir}. Run nas/compare-track.py first."
        )
    return candidates[-1].parent

RUN_DIR_OVERRIDE: Path | str | None = "f926b9"  # e.g., "compare_20260410T133741"
# RUN_DIR_OVERRIDE: Path | str | None = None


def resolve_run_dir() -> Path:
    if RUN_DIR_OVERRIDE:
        candidate = Path(RUN_DIR_OVERRIDE)
        if not candidate.is_absolute():
            candidate = BASE_OUTPUT_DIR / candidate
        if candidate.is_file():
            candidate = candidate.parent
        if not candidate.exists():
            raise FileNotFoundError(
                f"Override directory {candidate} was not found. Update RUN_DIR_OVERRIDE."
            )
        metrics_file = candidate / "metrics.jsonl"
        if not metrics_file.exists():
            raise FileNotFoundError(
                f"{metrics_file} was not found; choose a directory containing metrics.jsonl."
            )
        return candidate
    return latest_run_dir()

RUN_DIR = resolve_run_dir()
METRICS_FILE = RUN_DIR / "metrics.jsonl"
print(f"Using metrics from: {RUN_DIR}")

with METRICS_FILE.open("r", encoding="utf-8") as fh:
    METRIC_RECORDS = [json.loads(line) for line in fh if line.strip()]

if not METRIC_RECORDS:
    raise RuntimeError(f"{METRICS_FILE} is empty; rerun compare-track.")

def display_map_name(map_name: str) -> str:
    return Path(map_name).name


rows: list[dict[str, object]] = []
for record in METRIC_RECORDS:
    map_name = record["map"]
    for run in record["runs"]:
        rows.append(
            {
                "map": map_name,
                "map_label": display_map_name(map_name),
                "arch": run["label"],
                "rmse": run.get("rmse"),
            }
        )

df = pd.DataFrame(rows)
pivot = df.pivot_table(index="arch", columns="map_label", values="rmse")
pivot = pivot.sort_index()

def default_highlight_arch(index: pd.Index) -> str | None:
    for candidate in (RUN_DIR.name, "arch8"):
        if candidate in index:
            return candidate
    return None


HIGHLIGHT_ARCH = default_highlight_arch(pivot.index)
print(f"Loaded RMSE table with shape {pivot.shape} (arches x maps).")


## Metric Setup

Shared metric labels and compare-run discovery used by the box plots.


In [ ]:
metric_specs = {
    "crosstrack_rmse_m": {
        "title": "Cross-track RMSE",
        "ylabel": "Cross-track RMSE (m)",
        "table_title": "Cross-track RMSE (m)",
        "ylim_bottom": 0.0,
    },
    "crosstrack_mean_m": {
        "title": "Mean cross-track error",
        "ylabel": "Mean cross-track error (m)",
        "table_title": "Mean cross-track error (m)",
        "ylim_bottom": 0.0,
    },
    "crosstrack_std_m": {
        "title": "Cross-track variability",
        "ylabel": "Cross-track std (m)",
        "table_title": "Cross-track std (m)",
        "ylim_bottom": 0.0,
    },
    "crosstrack_max_m": {
        "title": "Maximum cross-track error",
        "ylabel": "Max cross-track error (m)",
        "table_title": "Max cross-track error (m)",
        "ylim_bottom": 0.0,
    },
    "heading_error_rmse_deg": {
        "title": "Heading-error RMSE",
        "ylabel": "Heading-error RMSE (deg)",
        "table_title": "Heading-error RMSE (deg)",
        "ylim_bottom": 0.0,
    },
    "heading_error_max_deg": {
        "title": "Maximum heading error",
        "ylabel": "Max heading error (deg)",
        "table_title": "Max heading error (deg)",
        "ylim_bottom": 0.0,
    },
    "wall_min_distance_m": {
        "title": "Minimum wall clearance",
        "ylabel": "Min wall distance (m)",
        "table_title": "Min wall distance (m)",
        "ylim_bottom": 0.0,
    },
    "wall_min_distance_risk": {
        "title": "Wall clearance risk",
        "ylabel": "Wall clearance risk (normalized)",
        "table_title": "Wall clearance risk (normalized)",
        "ylim_bottom": 0.0,
    },
    "steering_rate_mean_rad_s": {
        "title": "Mean steering-rate effort",
        "ylabel": "Mean steering rate (rad/s)",
        "table_title": "Mean steering rate (rad/s)",
        "ylim_bottom": 0.0,
    },
    "steering_rate_std_rad_s": {
        "title": "Steering-rate variability",
        "ylabel": "Steering-rate std (rad/s)",
        "table_title": "Steering-rate std (rad/s)",
        "ylim_bottom": 0.0,
    },
    "steering_rate_max_rad_s": {
        "title": "Maximum steering-rate spike",
        "ylabel": "Max steering rate (rad/s)",
        "table_title": "Max steering rate (rad/s)",
        "ylim_bottom": 0.0,
    },
    "speed_mean_m_s": {
        "title": "Mean speed",
        "ylabel": "Mean speed (m/s)",
        "table_title": "Mean speed (m/s)",
        "ylim_bottom": 0.0,
    },
    "speed_std_m_s": {
        "title": "Speed variability",
        "ylabel": "Speed std (m/s)",
        "table_title": "Speed std (m/s)",
        "ylim_bottom": 0.0,
    },
}


In [ ]:
EXCLUDE_COMPARE_RUN_IDS = {
    "08e72b",  # Moscow Raceway
    "3d2630",  # Oschersleben
    "782a1c",  # Montreal
    "a8b20f",  # Spielberg
    "e6278a",  # Hockenheim
    "d24a66",  # Zandvoort
}
TRAINED_MAP_LABELS = {
    "017f7c": "Yas Marina",
    "08e72b": "Moscow Raceway",
    "193215": "Melbourne",
    "1b391f": "Sakhir",
    "3cd867": "Budapest",
    "3d2630": "Oschersleben",
    "4ec05d": "IMS",
    "60de23": "Sao Paulo",
    "782a1c": "Montreal",
    "a8b20f": "Spielberg",
    "be986b": "Austin",
    "d24a66": "Zandvoort",
    "dc559d": "Catalunya",
    "e6278a": "Hockenheim",
    "f847ab": "Brands Hatch",
    "f926b9": "Sepang",
}

def discover_compare_run_dirs(base_dir: Path = BASE_OUTPUT_DIR) -> list[Path]:
    run_dirs = sorted(
        path.parent
        for path in base_dir.glob("*/metrics.jsonl")
        if path.parent.name not in EXCLUDE_COMPARE_RUN_IDS
    )
    if not run_dirs:
        raise FileNotFoundError(f"No metrics.jsonl run directories found under {base_dir}")
    return run_dirs


COMPARE_RUN_DIRS = discover_compare_run_dirs()
print("Compare runs:", ", ".join(path.name for path in COMPARE_RUN_DIRS))
CROSS_RUN_AGG = "mean"


def resolve_compare_run_dir(path_like: str | Path) -> Path:
    path = Path(path_like)
    if not path.is_absolute():
        path = BASE_OUTPUT_DIR / path
    if path.is_file():
        path = path.parent
    metrics_path = path / "metrics.jsonl"
    if not metrics_path.exists():
        raise FileNotFoundError(f"{metrics_path} does not exist")
    return path


def load_run_metric_rows(run_dir: Path, metric_key: str) -> pd.DataFrame:
    source_metric = "wall_min_distance_m" if metric_key == "wall_min_distance_risk" else metric_key
    metrics_path = run_dir / "metrics.jsonl"
    rows: list[dict[str, object]] = []
    with metrics_path.open("r", encoding="utf-8") as fh:
        records = [json.loads(line) for line in fh if line.strip()]

    for record in records:
        for run in record["runs"]:
            raw_label = run["label"]
            arch_label = "arch8" if raw_label == run_dir.name else raw_label
            rows.append(
                {
                    "run_dir": run_dir.name,
                    "map": record["map"],
                    "map_label": display_map_name(record["map"]),
                    "arch": arch_label,
                    "raw_label": raw_label,
                    "value": run.get(source_metric),
                }
            )
    return pd.DataFrame(rows)


def apply_derived_metric_transform(metric_df: pd.DataFrame, metric_key: str) -> pd.DataFrame:
    if metric_key != "wall_min_distance_risk":
        return metric_df
    metric_df = metric_df.copy()
    span = metric_df["value"].max() - metric_df["value"].min()
    if span == 0:
        metric_df["value"] = 0.5
    else:
        metric_df["value"] = 1.0 - ((metric_df["value"] - metric_df["value"].min()) / span)
    return metric_df


def aggregate_cross_run_metric(run_dirs: list[str | Path], metric_key: str) -> pd.DataFrame:
    frames = [load_run_metric_rows(resolve_compare_run_dir(run_dir), metric_key) for run_dir in run_dirs]
    metric_df = pd.concat(frames, ignore_index=True)
    metric_df["value"] = pd.to_numeric(metric_df["value"], errors="coerce")
    metric_df = metric_df.dropna(subset=["value"])
    metric_df = apply_derived_metric_transform(metric_df, metric_key)

    if CROSS_RUN_AGG == "mean":
        return metric_df.groupby(["run_dir", "arch"], as_index=False)["value"].mean()
    if CROSS_RUN_AGG == "median":
        return metric_df.groupby(["run_dir", "arch"], as_index=False)["value"].median()
    raise ValueError(f"Unsupported CROSS_RUN_AGG={CROSS_RUN_AGG!r}")


## Cross-Run Metric Box Plots

Each x-axis group is a training map. For each architecture, the box-and-whisker plot summarizes that metric across the evaluation maps in the corresponding `metrics.jsonl`.


In [ ]:
def cross_run_metric_values(run_dirs: list[str | Path], metric_key: str) -> pd.DataFrame:
    frames = [load_run_metric_rows(resolve_compare_run_dir(run_dir), metric_key) for run_dir in run_dirs]
    metric_df = pd.concat(frames, ignore_index=True)
    metric_df["value"] = pd.to_numeric(metric_df["value"], errors="coerce")
    metric_df = metric_df.dropna(subset=["value"])
    return apply_derived_metric_transform(metric_df, metric_key)


def plot_cross_run_metric_boxplots(run_dirs: list[str | Path], metric_key: str) -> None:
    metric_df = cross_run_metric_values(run_dirs, metric_key)
    resolved_dirs = [resolve_compare_run_dir(run_dir) for run_dir in run_dirs]
    run_labels = [run_dir.name for run_dir in resolved_dirs]
    x_labels = [TRAINED_MAP_LABELS.get(run_dir.name, run_dir.name) for run_dir in resolved_dirs]
    x_lookup = {label: idx for idx, label in enumerate(run_labels)}
    arch_order = [f"arch{i}" for i in range(1, 9)]
    arch_offsets = {arch: offset for arch, offset in zip(arch_order, np.linspace(-0.28, 0.28, len(arch_order)))}

    fig_width = max(12, len(run_labels) * 1.05)
    fig = plt.figure(figsize=(fig_width, 8.5), constrained_layout=True)
    grid = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[0.13, 0.87])
    legend_ax = fig.add_subplot(grid[0])
    ax = fig.add_subplot(grid[1])
    legend_ax.axis("off")

    handles = []
    labels = []
    for arch in arch_order:
        positions = []
        values = []
        for run_label in run_labels:
            series = metric_df[(metric_df["run_dir"] == run_label) & (metric_df["arch"] == arch)]["value"].dropna().to_numpy()
            if series.size == 0:
                continue
            positions.append(x_lookup[run_label] + arch_offsets[arch])
            values.append(series)
        if not values:
            continue

        color = "#d62728" if arch == "arch8" else plt.cm.Greys(0.25 + 0.55 * (int(arch[4:]) - 1) / 6)
        box = ax.boxplot(
            values,
            positions=positions,
            widths=0.06,
            patch_artist=True,
            manage_ticks=False,
            showfliers=False,
            boxprops={"facecolor": color, "edgecolor": color, "alpha": 0.45},
            medianprops={"color": color, "linewidth": 1.8},
            whiskerprops={"color": color, "linewidth": 1.1},
            capprops={"color": color, "linewidth": 1.1},
        )
        handles.append(box["boxes"][0])
        labels.append(arch)

    metric_label = metric_specs.get(metric_key, {}).get("ylabel", metric_key)
    ax.set_xticks(range(len(run_labels)))
    ax.set_xticklabels(x_labels, rotation=35, ha="right", rotation_mode="anchor")
    ax.tick_params(axis="x", labelsize=9, pad=4)
    ax.set_ylabel(metric_label, labelpad=8)
    ax.set_xlabel("Training map", labelpad=10)
    ax.grid(True, axis="y", linestyle="--", alpha=0.35)

    fig.suptitle(f"{metric_label} distribution by training map", y=0.995)
    legend_ax.legend(
        handles,
        labels,
        loc="center",
        ncol=len(labels),
        frameon=True,
        framealpha=0.95,
        handletextpad=0.35,
        columnspacing=1.1,
        borderpad=0.45,
        fontsize=9,
    )
    plt.show()


for metric_key in metric_specs:
    plot_cross_run_metric_boxplots(COMPARE_RUN_DIRS, metric_key)


## Composite Safety Metric Box Plot

This plot combines safety-relevant metrics where lower values are better after min-max normalizing each metric across the selected run directories.


In [ ]:
COMPOSITE_EXCLUDE_ARCHES = [
    # "arch1",
    "arch6",
    # "arch2",
]

COMPOSITE_EXCLUDE_RUN_IDS = [
    "08e72b",  # Moscow Raceway
    "3d2630",  # Oschersleben
    "782a1c",  # Montreal
    "a8b20f",  # Spielberg
    "e6278a",  # Hockenheim
    "d24a66",  # Zandvoort
]
COMPOSITE_RUN_DIRS = [
    run_dir for run_dir in COMPARE_RUN_DIRS
    if run_dir.name not in set(COMPOSITE_EXCLUDE_RUN_IDS)
]
print("Composite runs:", ", ".join(run_dir.name for run_dir in COMPOSITE_RUN_DIRS))

SAFETY_MAXIMIZE_METRICS = [
    "wall_min_distance_m",
]

# Only do one crosstrack, one heading error, one steering rate, one speed, 
SAFETY_MINIMIZE_METRICS = [
    # "crosstrack_rmse_m",
    # "crosstrack_mean_m",
    # "crosstrack_std_m",
    "crosstrack_max_m",
    # "heading_error_rmse_deg",
    "heading_error_max_deg",
    "wall_min_distance_risk",
    "steering_rate_mean_rad_s",
    # "steering_rate_std_rad_s",
    # "steering_rate_max_rad_s",
    "speed_mean_m_s",
    # "speed_std_m_s",
]


def normalize_metric(series: pd.Series) -> pd.Series:
    span = series.max() - series.min()
    if span == 0:
        return pd.Series(0.5, index=series.index)
    return (series - series.min()) / span


def composite_metric_values(run_dirs: list[str | Path], metric_keys: list[str]) -> pd.DataFrame:
    excluded_arches = (
        {COMPOSITE_EXCLUDE_ARCHES}
        if isinstance(COMPOSITE_EXCLUDE_ARCHES, str)
        else set(COMPOSITE_EXCLUDE_ARCHES)
    )
    frames = []
    for metric_key in metric_keys:
        metric_df = cross_run_metric_values(run_dirs, metric_key).copy()
        if excluded_arches:
            metric_df = metric_df[~metric_df["arch"].isin(excluded_arches)]
        metric_df["metric"] = metric_key
        metric_df["normalized_value"] = normalize_metric(metric_df["value"])
        frames.append(metric_df)
    return pd.concat(frames, ignore_index=True)


def plot_composite_metric_boxplots(
    run_dirs: list[str | Path],
    metric_keys: list[str],
    title: str,
    ylabel: str,
) -> None:
    metric_df = composite_metric_values(run_dirs, metric_keys)
    resolved_dirs = [resolve_compare_run_dir(run_dir) for run_dir in run_dirs]
    run_labels = [run_dir.name for run_dir in resolved_dirs]
    x_labels = [TRAINED_MAP_LABELS.get(run_dir.name, run_dir.name) for run_dir in resolved_dirs]
    x_lookup = {label: idx for idx, label in enumerate(run_labels)}
    all_arches = [f"arch{i}" for i in range(1, 9)]
    arch_order = [arch for arch in all_arches if arch in set(metric_df["arch"])]
    baseline_arches = [arch for arch in arch_order if arch != "arch8"]
    arch_display_labels = {
        arch: f"Architecture {chr(ord('A') + idx)}"
        for idx, arch in enumerate(baseline_arches)
    }
    if "arch8" in arch_order:
        arch_display_labels["arch8"] = "Architecture A*"
    arch_count = max(len(arch_order), 1)
    group_span = 0.09 * max(arch_count - 1, 0)
    arch_offset_values = [0.0] if arch_count == 1 else np.linspace(-group_span / 2, group_span / 2, arch_count)
    arch_offsets = {arch: offset for arch, offset in zip(arch_order, arch_offset_values)}
    box_width = 0.06

    fig_width = max(12, len(run_labels) * 1.05)
    fig = plt.figure(figsize=(fig_width, 6.5), constrained_layout=True)
    grid = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[0.16, 0.84])
    legend_ax = fig.add_subplot(grid[0])
    ax = fig.add_subplot(grid[1])
    legend_ax.axis("off")

    handles = []
    labels = []
    for arch in arch_order:
        positions = []
        values = []
        for run_label in run_labels:
            series = metric_df[(metric_df["run_dir"] == run_label) & (metric_df["arch"] == arch)]["normalized_value"].dropna().to_numpy()
            if series.size == 0:
                continue
            positions.append(x_lookup[run_label] + arch_offsets[arch])
            values.append(series)
        if not values:
            continue

        color = "#d62728" if arch == "arch8" else plt.cm.Greys(0.25 + 0.55 * (int(arch[4:]) - 1) / 6)
        box = ax.boxplot(
            values,
            positions=positions,
            widths=box_width,
            patch_artist=True,
            manage_ticks=False,
            showfliers=False,
            boxprops={"facecolor": color, "edgecolor": color, "alpha": 0.45},
            medianprops={"color": color, "linewidth": 1.8},
            whiskerprops={"color": color, "linewidth": 1.1},
            capprops={"color": color, "linewidth": 1.1},
        )
        handles.append(box["boxes"][0])
        labels.append(arch_display_labels[arch])

    ax.set_xticks(range(len(run_labels)))
    ax.set_xticklabels(x_labels, rotation=35, ha="right", rotation_mode="anchor")
    ax.tick_params(axis="x", labelsize=9, pad=4)
    ax.set_ylabel(ylabel, labelpad=8)
    ax.set_xlabel("Training map", labelpad=10)
    ax.grid(True, axis="y", linestyle="--", alpha=0.35)

    fig.suptitle(title, y=0.995)
    legend_ax.legend(
        handles,
        labels,
        loc="center",
        ncol=max(1, len(labels)),
        frameon=True,
        framealpha=0.95,
        handletextpad=0.35,
        columnspacing=1.1,
        borderpad=0.45,
        fontsize=9,
    )
    output_dir = BASE_OUTPUT_DIR.parents[1] if BASE_OUTPUT_DIR.parent.name == "nas" else Path.cwd()
    pdf_path = output_dir / "composite-metrics.pdf"
    svg_path = output_dir / "composite-metrics.svg"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight")
    print(f"Saved composite metrics plot to {pdf_path}")
    print(f"Saved editable composite metrics plot to {svg_path}")
    plt.show()


plot_composite_metric_boxplots(
    COMPOSITE_RUN_DIRS,
    SAFETY_MINIMIZE_METRICS,
    title="Composite safety-cost metrics",
    ylabel="Normalized safety-cost value",
)


## Composite Safety Improvement Summary

This compares arch8 against the pooled arch1-7 baseline using the same composite metric values plotted above.


In [ ]:
def percent_lower_than(reference: float, candidate: float) -> float:
    if pd.isna(reference) or pd.isna(candidate) or reference == 0:
        return np.nan
    return 100.0 * (reference - candidate) / reference


def summarize_distribution(series: pd.Series) -> pd.Series:
    return pd.Series(
        {
            "mean_score": series.mean(),
            "median_score": series.median(),
            "q1_score": series.quantile(0.25),
            "q3_score": series.quantile(0.75),
            "min_score": series.min(),
            "max_score": series.max(),
            "n_values": series.count(),
        }
    )


resolved_composite_dirs = [resolve_compare_run_dir(run_dir) for run_dir in COMPOSITE_RUN_DIRS]
print("Included composite runs:", ", ".join(run_dir.name for run_dir in resolved_composite_dirs))

safety_values = composite_metric_values(resolved_composite_dirs, SAFETY_MINIMIZE_METRICS).copy()
composite_excluded_arches = (
    {COMPOSITE_EXCLUDE_ARCHES}
    if isinstance(COMPOSITE_EXCLUDE_ARCHES, str)
    else set(COMPOSITE_EXCLUDE_ARCHES)
)
composite_baseline_arches = [
    f"arch{i}" for i in range(1, 8)
    if f"arch{i}" not in composite_excluded_arches
]
composite_baseline_label = ", ".join(composite_baseline_arches) if composite_baseline_arches else "baseline"
safety_values["group"] = np.where(safety_values["arch"] == "arch8", "arch8", composite_baseline_label)
print(
    "Composite values:",
    f"{(safety_values['arch'] == 'arch8').sum()} arch8 values and",
    f"{safety_values['arch'].isin(composite_baseline_arches).sum()} {composite_baseline_label} baseline values",
)

pooled_summary = (
    safety_values[safety_values["arch"].isin([f"arch{i}" for i in range(1, 9)])]
    .groupby("group")["normalized_value"]
    .apply(summarize_distribution)
    .unstack()
)

arch8_pooled = pooled_summary.loc["arch8"]
baseline_pooled = pooled_summary.loc[composite_baseline_label]
pooled_mean_improvement = percent_lower_than(baseline_pooled["mean_score"], arch8_pooled["mean_score"])
pooled_q3_improvement = percent_lower_than(baseline_pooled["q3_score"], arch8_pooled["q3_score"])

print("Composite safety-cost summary")
print(
    "Across the full pooled distribution, arch8 is "
    f"{pooled_mean_improvement:+.1f}% lower than the pooled {composite_baseline_label} baseline."
)
print(
    f"Mean scores: arch8 {arch8_pooled['mean_score']:.3f} "
    f"vs baseline {baseline_pooled['mean_score']:.3f}."
)
print(f"Q3 improvement: {pooled_q3_improvement:+.1f}%.")

rows = []
for run_dir, run_df in safety_values.groupby("run_dir"):
    arch8_values = run_df[run_df["arch"] == "arch8"]["normalized_value"]
    baseline_values = run_df[run_df["arch"].isin(composite_baseline_arches)]["normalized_value"]

    arch8_summary = summarize_distribution(arch8_values)
    baseline_summary = summarize_distribution(baseline_values)
    baseline_arch_scores = (
        run_df[run_df["arch"].isin(composite_baseline_arches)]
        .groupby("arch")["normalized_value"]
        .mean()
    )
    best_baseline_arch = baseline_arch_scores.idxmin()
    best_baseline_score = baseline_arch_scores.min()

    rows.append(
        {
            "training_map": TRAINED_MAP_LABELS.get(run_dir, run_dir),
            "run_dir": run_dir,
            "arch8_mean_score": arch8_summary["mean_score"],
            "baseline_mean_score": baseline_summary["mean_score"],
            "arch8_vs_baseline_mean_%": percent_lower_than(
                baseline_summary["mean_score"], arch8_summary["mean_score"]
            ),
            "arch8_q3_score": arch8_summary["q3_score"],
            "baseline_q3_score": baseline_summary["q3_score"],
            "arch8_vs_baseline_q3_%": percent_lower_than(
                baseline_summary["q3_score"], arch8_summary["q3_score"]
            ),
            "best_baseline_arch": best_baseline_arch,
            "best_baseline_mean_score": best_baseline_score,
            "arch8_vs_best_baseline_mean_%": percent_lower_than(
                best_baseline_score, arch8_summary["mean_score"]
            ),
            "n_arch8_values": arch8_summary["n_values"],
            "n_baseline_values": baseline_summary["n_values"],
        }
    )

safety_improvement_by_map = pd.DataFrame(rows)

display(
    safety_improvement_by_map.style.format(
        {
            "arch8_mean_score": "{:.3f}",
            "baseline_mean_score": "{:.3f}",
            "arch8_vs_baseline_mean_%": "{:+.1f}%",
            "arch8_q3_score": "{:.3f}",
            "baseline_q3_score": "{:.3f}",
            "arch8_vs_baseline_q3_%": "{:+.1f}%",
            "best_baseline_mean_score": "{:.3f}",
            "arch8_vs_best_baseline_mean_%": "{:+.1f}%",
            "n_arch8_values": "{:.0f}",
            "n_baseline_values": "{:.0f}",
        }
    )
)


In [ ]:
SELECTED_IMPROVEMENT_METRICS = [
    # "crosstrack_rmse_m",
    # "crosstrack_mean_m",
    # "crosstrack_std_m",
    "crosstrack_max_m",
    # "heading_error_rmse_deg",
    "heading_error_max_deg",
    "wall_min_distance_risk",
    "steering_rate_mean_rad_s",
    # "steering_rate_std_rad_s",
    # "steering_rate_max_rad_s",
    "speed_mean_m_s",
    # "speed_std_m_s",
]

selected_values = []
excluded_composite_arches = (
    {COMPOSITE_EXCLUDE_ARCHES}
    if isinstance(COMPOSITE_EXCLUDE_ARCHES, str)
    else set(COMPOSITE_EXCLUDE_ARCHES)
)
included_baseline_arches = [
    f"arch{i}" for i in range(1, 8)
    if f"arch{i}" not in excluded_composite_arches
]
included_arches = included_baseline_arches + ["arch8"]
baseline_label = ", ".join(included_baseline_arches) if included_baseline_arches else "baseline"

for metric_key in SELECTED_IMPROVEMENT_METRICS:
    metric_df = cross_run_metric_values(resolved_composite_dirs, metric_key).copy()
    metric_df["metric"] = metric_key
    metric_df["metric_label"] = metric_specs.get(metric_key, {}).get("ylabel", metric_key)
    metric_df = metric_df[metric_df["arch"].isin(included_arches)]
    metric_df["normalized_value"] = normalize_metric(metric_df["value"])
    metric_df["group"] = np.where(metric_df["arch"] == "arch8", "arch8", baseline_label)
    selected_values.append(metric_df)

selected_metric_values = pd.concat(selected_values, ignore_index=True)

print("Included runs:", ", ".join(run_dir.name for run_dir in resolved_composite_dirs))
print("Included baseline arches:", baseline_label)

rows = []
for metric_key, metric_df in selected_metric_values.groupby("metric", sort=False):
    arch8_values = metric_df[metric_df["arch"] == "arch8"]["normalized_value"]
    baseline_values = metric_df[metric_df["arch"].isin(included_baseline_arches)]["normalized_value"]
    arch8_summary = summarize_distribution(arch8_values)
    baseline_summary = summarize_distribution(baseline_values)
    baseline_arch_scores = (
        metric_df[metric_df["arch"].isin(included_baseline_arches)]
        .groupby("arch")["normalized_value"]
        .mean()
    )
    best_baseline_arch = baseline_arch_scores.idxmin()
    best_baseline_score = baseline_arch_scores.min()

    rows.append(
        {
            "metric": metric_specs.get(metric_key, {}).get("ylabel", metric_key),
            "metric_key": metric_key,
            "arch8_mean": arch8_summary["mean_score"],
            "baseline_mean": baseline_summary["mean_score"],
            "arch8_vs_baseline_mean_%": percent_lower_than(
                baseline_summary["mean_score"], arch8_summary["mean_score"]
            ),
            "arch8_q3": arch8_summary["q3_score"],
            "baseline_q3": baseline_summary["q3_score"],
            "arch8_vs_baseline_q3_%": percent_lower_than(
                baseline_summary["q3_score"], arch8_summary["q3_score"]
            ),
            "best_baseline_arch": best_baseline_arch,
            "best_baseline_mean": best_baseline_score,
            "arch8_vs_best_baseline_mean_%": percent_lower_than(
                best_baseline_score, arch8_summary["mean_score"]
            ),
            "n_arch8_values": arch8_summary["n_values"],
            "n_baseline_values": baseline_summary["n_values"],
        }
    )

selected_metric_improvement_summary = pd.DataFrame(rows)

print("Selected metric improvement summaries")
for row in rows:
    print(row["metric"])
    print(
        "Across the full pooled distribution, arch8 is "
        f"{row['arch8_vs_baseline_mean_%']:+.1f}% lower than the pooled {baseline_label} baseline."
    )
    print(
        f"Normalized mean scores: arch8 {row['arch8_mean']:.3f} "
        f"vs baseline {row['baseline_mean']:.3f}."
    )
    print(f"Q3 improvement: {row['arch8_vs_baseline_q3_%']:+.1f}%.")
    print()

display(
    selected_metric_improvement_summary.style.format(
        {
            "arch8_mean": "{:.3f}",
            "baseline_mean": "{:.3f}",
            "arch8_vs_baseline_mean_%": "{:+.1f}%",
            "arch8_q3": "{:.3f}",
            "baseline_q3": "{:.3f}",
            "arch8_vs_baseline_q3_%": "{:+.1f}%",
            "best_baseline_mean": "{:.3f}",
            "arch8_vs_best_baseline_mean_%": "{:+.1f}%",
            "n_arch8_values": "{:.0f}",
            "n_baseline_values": "{:.0f}",
        }
    )
)

rows = []
for (metric_key, run_dir), run_df in selected_metric_values.groupby(["metric", "run_dir"], sort=False):
    arch8_values = run_df[run_df["arch"] == "arch8"]["normalized_value"]
    baseline_values = run_df[run_df["arch"].isin(included_baseline_arches)]["normalized_value"]
    arch8_summary = summarize_distribution(arch8_values)
    baseline_summary = summarize_distribution(baseline_values)

    rows.append(
        {
            "metric": metric_specs.get(metric_key, {}).get("ylabel", metric_key),
            "training_map": TRAINED_MAP_LABELS.get(run_dir, run_dir),
            "run_dir": run_dir,
            "arch8_mean": arch8_summary["mean_score"],
            "baseline_mean": baseline_summary["mean_score"],
            "arch8_vs_baseline_mean_%": percent_lower_than(
                baseline_summary["mean_score"], arch8_summary["mean_score"]
            ),
            "arch8_q3": arch8_summary["q3_score"],
            "baseline_q3": baseline_summary["q3_score"],
            "arch8_vs_baseline_q3_%": percent_lower_than(
                baseline_summary["q3_score"], arch8_summary["q3_score"]
            ),
            "n_arch8_values": arch8_summary["n_values"],
            "n_baseline_values": baseline_summary["n_values"],
        }
    )

selected_metric_improvement_by_map = pd.DataFrame(rows)

display(
    selected_metric_improvement_by_map.style.format(
        {
            "arch8_mean": "{:.3f}",
            "baseline_mean": "{:.3f}",
            "arch8_vs_baseline_mean_%": "{:+.1f}%",
            "arch8_q3": "{:.3f}",
            "baseline_q3": "{:.3f}",
            "arch8_vs_baseline_q3_%": "{:+.1f}%",
            "n_arch8_values": "{:.0f}",
            "n_baseline_values": "{:.0f}",
        }
    )
)


In [ ]:
COMPONENT_IMPROVEMENT_METRICS = {
    "CTE": "crosstrack_max_m",
    "Heading": "heading_error_max_deg",
    "Clearance": "wall_min_distance_risk",
    "Steering": "steering_rate_mean_rad_s",
    "Speed": "speed_mean_m_s",
}

component_excluded_run_ids = (
    {COMPOSITE_EXCLUDE_RUN_IDS}
    if isinstance(COMPOSITE_EXCLUDE_RUN_IDS, str)
    else set(COMPOSITE_EXCLUDE_RUN_IDS)
)
component_excluded_arches = (
    {COMPOSITE_EXCLUDE_ARCHES}
    if isinstance(COMPOSITE_EXCLUDE_ARCHES, str)
    else set(COMPOSITE_EXCLUDE_ARCHES)
)
component_baseline_arches = [
    f"arch{i}" for i in range(1, 8)
    if f"arch{i}" not in component_excluded_arches
]
component_included_arches = component_baseline_arches + ["arch8"]
component_baseline_label = ", ".join(component_baseline_arches) if component_baseline_arches else "baseline"
component_run_dirs = [
    run_dir for run_dir in COMPARE_RUN_DIRS
    if run_dir.name not in component_excluded_run_ids
]
resolved_component_dirs = [resolve_compare_run_dir(run_dir) for run_dir in component_run_dirs]

component_frames = []
for component, metric_key in COMPONENT_IMPROVEMENT_METRICS.items():
    metric_df = cross_run_metric_values(resolved_component_dirs, metric_key).copy()
    metric_df = metric_df[metric_df["arch"].isin(component_included_arches)]
    metric_df["component"] = component
    metric_df["metric"] = metric_key
    metric_df["normalized_value"] = normalize_metric(metric_df["value"])
    component_frames.append(metric_df)

component_values = pd.concat(component_frames, ignore_index=True)

rows = []
for (map_label, component), track_df in component_values.groupby(["map_label", "component"], sort=False):
    arch8_values = track_df[track_df["arch"] == "arch8"]["normalized_value"]
    baseline_values = track_df[track_df["arch"].isin(component_baseline_arches)]["normalized_value"]
    arch8_mean = arch8_values.mean()
    baseline_mean = baseline_values.mean()

    rows.append(
        {
            "track": map_label,
            "component": component,
            "arch8_mean": arch8_mean,
            "baseline_mean": baseline_mean,
            "arch8_vs_baseline_mean_%": percent_lower_than(baseline_mean, arch8_mean),
            "n_arch8_values": arch8_values.count(),
            "n_baseline_values": baseline_values.count(),
        }
    )

component_improvement_by_track = pd.DataFrame(rows)
component_order = list(COMPONENT_IMPROVEMENT_METRICS)

component_improvement_track_pivot = (
    component_improvement_by_track
    .pivot(index="track", columns="component", values="arch8_vs_baseline_mean_%")
    .reindex(columns=component_order)
)
component_improvement_mean_across_tracks = (
    component_improvement_track_pivot
    .mean(axis=0)
    .rename("Mean")
    .to_frame()
    .T
)

print("Included training tracks:", ", ".join(TRAINED_MAP_LABELS.get(run_dir.name, run_dir.name) for run_dir in resolved_component_dirs))
if component_excluded_run_ids:
    print("Excluded training tracks:", ", ".join(TRAINED_MAP_LABELS.get(run_id, run_id) for run_id in sorted(component_excluded_run_ids)))
print("Per-track mean percent improvement for Architecture A* vs", component_baseline_label)
display(component_improvement_track_pivot.style.format("{:+.1f}%"))

print("Mean of each component across evaluated tracks")
display(component_improvement_mean_across_tracks.style.format("{:+.1f}%"))

display(
    component_improvement_by_track.style.format(
        {
            "arch8_mean": "{:.3f}",
            "baseline_mean": "{:.3f}",
            "arch8_vs_baseline_mean_%": "{:+.1f}%",
            "n_arch8_values": "{:.0f}",
            "n_baseline_values": "{:.0f}",
        }
    )
)


In [ ]:
# Mean normalized Architecture A* component values by training track.
# Rows are training tracks, columns are component means over all evaluated testing tracks.

component_excluded_run_ids = (
    {COMPOSITE_EXCLUDE_RUN_IDS}
    if isinstance(COMPOSITE_EXCLUDE_RUN_IDS, str)
    else set(COMPOSITE_EXCLUDE_RUN_IDS)
)
component_excluded_arches = (
    {COMPOSITE_EXCLUDE_ARCHES}
    if isinstance(COMPOSITE_EXCLUDE_ARCHES, str)
    else set(COMPOSITE_EXCLUDE_ARCHES)
)
component_baseline_arches = [
    f"arch{i}" for i in range(1, 8)
    if f"arch{i}" not in component_excluded_arches
]
component_included_arches = component_baseline_arches + ["arch8"]
component_baseline_label = ", ".join(component_baseline_arches) if component_baseline_arches else "baseline"
component_run_dirs = [
    run_dir for run_dir in COMPARE_RUN_DIRS
    if run_dir.name not in component_excluded_run_ids
]
resolved_component_dirs = [resolve_compare_run_dir(run_dir) for run_dir in component_run_dirs]

component_value_frames = []
for component, metric_key in COMPONENT_IMPROVEMENT_METRICS.items():
    metric_df = cross_run_metric_values(resolved_component_dirs, metric_key).copy()
    metric_df = metric_df[metric_df["arch"].isin(component_included_arches)]
    metric_df["component"] = component
    metric_df["metric"] = metric_key
    metric_df["normalized_value"] = normalize_metric(metric_df["value"])
    component_value_frames.append(metric_df)

training_component_values = pd.concat(component_value_frames, ignore_index=True)

arch8_training_component_means = (
    training_component_values[training_component_values["arch"] == "arch8"]
    .groupby(["run_dir", "component"], as_index=False)["normalized_value"]
    .mean()
)
arch8_training_component_means["training_track"] = arch8_training_component_means["run_dir"].map(
    lambda run_dir: TRAINED_MAP_LABELS.get(run_dir, run_dir)
)

arch8_training_component_table = (
    arch8_training_component_means
    .pivot(index="training_track", columns="component", values="normalized_value")
    .reindex(columns=component_order)
)

net_improvement_by_component = []
for component, component_df in training_component_values.groupby("component", sort=False):
    arch8_mean = component_df.loc[component_df["arch"] == "arch8", "normalized_value"].mean()
    baseline_mean = component_df.loc[
        component_df["arch"].isin(component_baseline_arches), "normalized_value"
    ].mean()
    net_improvement_by_component.append(
        {
            "component": component,
            "arch8_mean": arch8_mean,
            "baseline_mean": baseline_mean,
            "net_improvement_%": percent_lower_than(baseline_mean, arch8_mean),
        }
    )

net_improvement_by_component = pd.DataFrame(net_improvement_by_component).set_index("component")

baseline_component_row_label = f"Mean of included archs ({component_baseline_label})"
net_improvement_row_label = f"Net improvement vs {component_baseline_label} (%)"

arch8_training_component_table_with_net = arch8_training_component_table.copy()
arch8_training_component_table_with_net.loc[baseline_component_row_label] = (
    net_improvement_by_component["baseline_mean"].reindex(component_order)
)
arch8_training_component_table_with_net.loc[net_improvement_row_label] = (
    net_improvement_by_component["net_improvement_%"].reindex(component_order)
)

formatted_arch8_training_component_table = arch8_training_component_table_with_net.astype(object).copy()
for row_label in formatted_arch8_training_component_table.index:
    if row_label == net_improvement_row_label:
        formatted_arch8_training_component_table.loc[row_label] = formatted_arch8_training_component_table.loc[
            row_label
        ].map(lambda value: f"{value:+.1f}%")
    else:
        formatted_arch8_training_component_table.loc[row_label] = formatted_arch8_training_component_table.loc[
            row_label
        ].map(lambda value: f"{value:.3f}")

print("Included training tracks:", ", ".join(TRAINED_MAP_LABELS.get(run_dir.name, run_dir.name) for run_dir in resolved_component_dirs))
print("Included baseline architectures:", component_baseline_label)
if component_excluded_run_ids:
    print("Excluded training tracks:", ", ".join(TRAINED_MAP_LABELS.get(run_id, run_id) for run_id in sorted(component_excluded_run_ids)))
print("Mean normalized Architecture A* component values by training track")
display(formatted_arch8_training_component_table)

print("Numeric net-improvement details")
display(
    net_improvement_by_component.reindex(component_order).style.format(
        {
            "arch8_mean": "{:.3f}",
            "baseline_mean": "{:.3f}",
            "net_improvement_%": "{:+.1f}%",
        }
    )
)


In [ ]:
# Mean normalized composite-component values by architecture.
# Rows are architectures, columns are component means over included training and testing tracks.

architecture_component_excluded_run_ids = (
    {COMPOSITE_EXCLUDE_RUN_IDS}
    if isinstance(COMPOSITE_EXCLUDE_RUN_IDS, str)
    else set(COMPOSITE_EXCLUDE_RUN_IDS)
)
architecture_component_excluded_arches = (
    {COMPOSITE_EXCLUDE_ARCHES}
    if isinstance(COMPOSITE_EXCLUDE_ARCHES, str)
    else set(COMPOSITE_EXCLUDE_ARCHES)
)
architecture_component_arches = [
    f"arch{i}" for i in range(1, 8)
    if f"arch{i}" not in architecture_component_excluded_arches
] + ["arch8"]
architecture_component_run_dirs = [
    run_dir for run_dir in COMPARE_RUN_DIRS
    if run_dir.name not in architecture_component_excluded_run_ids
]
resolved_architecture_component_dirs = [
    resolve_compare_run_dir(run_dir) for run_dir in architecture_component_run_dirs
]

architecture_component_frames = []
for component, metric_key in COMPONENT_IMPROVEMENT_METRICS.items():
    metric_df = cross_run_metric_values(resolved_architecture_component_dirs, metric_key).copy()
    metric_df = metric_df[metric_df["arch"].isin(architecture_component_arches)]
    metric_df["component"] = component
    metric_df["metric"] = metric_key
    metric_df["normalized_value"] = normalize_metric(metric_df["value"])
    architecture_component_frames.append(metric_df)

architecture_component_values = pd.concat(architecture_component_frames, ignore_index=True)

architecture_component_means = (
    architecture_component_values
    .groupby(["arch", "component"], as_index=False)
    .agg(
        mean_normalized_value=("normalized_value", "mean"),
        n_values=("normalized_value", "count"),
    )
)

architecture_component_table = (
    architecture_component_means
    .pivot(index="arch", columns="component", values="mean_normalized_value")
    .reindex(index=architecture_component_arches, columns=component_order)
)

architecture_component_baseline_arches = [arch for arch in architecture_component_arches if arch != "arch8"]
architecture_component_baseline_mean = architecture_component_table.loc[
    architecture_component_baseline_arches
].mean(axis=0)
architecture_component_arch8_mean = architecture_component_table.loc["arch8"]
architecture_component_percent_difference = 100.0 * (
    architecture_component_baseline_mean - architecture_component_arch8_mean
) / architecture_component_baseline_mean

architecture_component_table_with_summary = architecture_component_table.copy()
architecture_component_table_with_summary.loc["Baseline mean"] = architecture_component_baseline_mean
architecture_component_table_with_summary.loc["Arch8 vs baseline (%)"] = architecture_component_percent_difference

architecture_component_display_table = architecture_component_table_with_summary.astype(object).copy()
for row_label in architecture_component_display_table.index:
    if row_label == "Arch8 vs baseline (%)":
        architecture_component_display_table.loc[row_label] = architecture_component_display_table.loc[
            row_label
        ].map(lambda value: f"{value:+.1f}%")
    else:
        architecture_component_display_table.loc[row_label] = architecture_component_display_table.loc[
            row_label
        ].map(lambda value: f"{value:.3f}")

architecture_component_counts = (
    architecture_component_means
    .pivot(index="arch", columns="component", values="n_values")
    .reindex(index=architecture_component_arches, columns=component_order)
)

print("Included training tracks:", ", ".join(TRAINED_MAP_LABELS.get(run_dir.name, run_dir.name) for run_dir in resolved_architecture_component_dirs))
if architecture_component_excluded_run_ids:
    print("Excluded training tracks:", ", ".join(TRAINED_MAP_LABELS.get(run_id, run_id) for run_id in sorted(architecture_component_excluded_run_ids)))
print("Included architectures:", ", ".join(architecture_component_arches))
print("Mean normalized composite-component values by architecture")
display(architecture_component_display_table)

print("Counts contributing to each mean")
display(architecture_component_counts.style.format("{:.0f}"))


In [ ]:
# Grouped BAW-style composite boxplot: baselines on the left, each included Architecture A* on the right.

def plot_baseline_vs_each_arch8_composite_boxplot(
    run_dirs: list[str | Path],
    metric_keys: list[str],
    title: str | None = None,
    ylabel: str = "Normalized Safety-Cost Value",
    save_stem: str | None = "composite-baseline-vs-each-arch8",
) -> None:
    plt.rcParams.update(
        {
            "font.family": "serif",
            "font.serif": ["Latin Modern Roman", "Times New Roman", "Times"],
            "font.size": 18,
            "axes.labelsize": 20,
            "axes.titlesize": 20,
            "xtick.labelsize": 14,
            "ytick.labelsize": 18,
            "mathtext.fontset": "cm",
        }
    )

    metric_df = composite_metric_values(run_dirs, metric_keys).copy()
    resolved_dirs = [resolve_compare_run_dir(run_dir) for run_dir in run_dirs]
    run_labels = [run_dir.name for run_dir in resolved_dirs]
    run_label_set = set(run_labels)
    metric_df = metric_df[metric_df["run_dir"].isin(run_label_set)]

    all_arches = [f"arch{i}" for i in range(1, 9)]
    present_arches = [arch for arch in all_arches if arch in set(metric_df["arch"])]
    baseline_arches = [arch for arch in present_arches if arch != "arch8"]
    arch8_run_labels = [run_label for run_label in run_labels if not metric_df[
        (metric_df["run_dir"] == run_label) & (metric_df["arch"] == "arch8")
    ].empty]

    baseline_display_labels = {
        arch: f"Architecture {chr(ord('A') + idx)}"
        for idx, arch in enumerate(baseline_arches)
    }
    arch8_display_labels = {
        run_label: rf"$\mathcal{{A}}^{{*}}$({TRAINED_MAP_LABELS.get(run_label, run_label)})"
        for run_label in arch8_run_labels
    }

    box_width = 0.40
    edge_gap = 0.30
    box_step = box_width + edge_gap
    baseline_positions = np.arange(len(baseline_arches), dtype=float) * box_step
    separator_x = (
        baseline_positions[-1] + box_width / 2 + edge_gap
        if len(baseline_positions)
        else 0.0
    )
    first_arch8_position = separator_x + edge_gap + box_width / 2
    arch8_positions = first_arch8_position + np.arange(len(arch8_run_labels), dtype=float) * box_step
    baseline_position_lookup = dict(zip(baseline_arches, baseline_positions))
    arch8_position_lookup = dict(zip(arch8_run_labels, arch8_positions))

    n_boxes = len(baseline_arches) + len(arch8_run_labels)
    fig_width = max(13, 0.85 * max(n_boxes, 1) + 3.0)
    fig, ax = plt.subplots(figsize=(fig_width, 6.4), constrained_layout=True)

    baseline_color = plt.cm.Greys(0.8)
    arch8_color = plt.cm.Blues(0.82)

    for arch in baseline_arches:
        values = metric_df.loc[metric_df["arch"] == arch, "normalized_value"].dropna().to_numpy()
        if values.size == 0:
            continue
        color = baseline_color
        box = ax.boxplot(
            [values],
            positions=[baseline_position_lookup[arch]],
            widths=box_width,
            patch_artist=True,
            manage_ticks=False,
            showfliers=False,
            boxprops={"facecolor": color, "edgecolor": color, "alpha": 0.58},
            medianprops={"color": color, "linewidth": 2.0},
            whiskerprops={"color": color, "linewidth": 1.2},
            capprops={"color": color, "linewidth": 1.2},
        )

    for run_label in arch8_run_labels:
        values = metric_df.loc[
            (metric_df["run_dir"] == run_label) & (metric_df["arch"] == "arch8"),
            "normalized_value",
        ].dropna().to_numpy()
        if values.size == 0:
            continue
        color = arch8_color
        box = ax.boxplot(
            [values],
            positions=[arch8_position_lookup[run_label]],
            widths=box_width,
            patch_artist=True,
            manage_ticks=False,
            showfliers=False,
            boxprops={"facecolor": color, "edgecolor": color, "alpha": 0.62},
            medianprops={"color": color, "linewidth": 2.0},
            whiskerprops={"color": color, "linewidth": 1.2},
            capprops={"color": color, "linewidth": 1.2},
        )

    if len(baseline_positions) and len(arch8_positions):
        ax.axvline(separator_x, color="0.25", linewidth=1.2, linestyle="--", alpha=0.75)
        ax.text(
            baseline_positions.mean(),
            1.02,
            "Baseline Architectures",
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="bottom",
            fontsize=20,
        )
        ax.text(
            arch8_positions.mean(),
            1.02,
            r"$\mathcal{A}^{*}$(training_track)",
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="bottom",
            fontsize=20,
            color="black",
        )

    tick_positions = [baseline_position_lookup[arch] for arch in baseline_arches] + [
        arch8_position_lookup[run_label] for run_label in arch8_run_labels
    ]
    tick_labels = [baseline_display_labels[arch] for arch in baseline_arches] + [
        arch8_display_labels[run_label] for run_label in arch8_run_labels
    ]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, rotation=35, ha="right", rotation_mode="anchor")
    ax.tick_params(axis="x", labelsize=14, pad=6)
    ax.tick_params(axis="y", labelsize=18)
    ax.set_ylabel(ylabel, labelpad=14)
    ax.set_xlabel("Architecture")
    if title:
        ax.set_title(title)
    ax.grid(True, axis="y", linestyle="--", alpha=0.35)
    all_positions = np.array(tick_positions, dtype=float)
    if all_positions.size:
        ax.set_xlim(
            all_positions.min() - box_width / 2 - edge_gap,
            all_positions.max() + box_width / 2 + edge_gap,
        )


    if save_stem:
        if BASE_OUTPUT_DIR.name == COMPARE_OUTPUT_DIR_NAME and BASE_OUTPUT_DIR.parent.name == "nas":
            save_root = BASE_OUTPUT_DIR.parent.parent
        else:
            save_root = Path.cwd().resolve()
        pdf_path = save_root / f"{save_stem}.pdf"
        svg_path = save_root / f"{save_stem}.svg"
        fig.savefig(pdf_path, bbox_inches="tight")
        fig.savefig(svg_path, bbox_inches="tight")
        print(f"Saved grouped composite plot to {pdf_path}")
        print(f"Saved editable grouped composite plot to {svg_path}")

    plt.show()


plot_baseline_vs_each_arch8_composite_boxplot(
    COMPOSITE_RUN_DIRS,
    SAFETY_MINIMIZE_METRICS,
)


In [ ]:
# Audit the math behind the normalized component table.
# This cell intentionally repeats the calculation in long form so each number can be traced.

component_math_frames = []
component_normalization_rows = []

for component, metric_key in COMPONENT_IMPROVEMENT_METRICS.items():
    raw_metric_key = "wall_min_distance_m" if metric_key == "wall_min_distance_risk" else metric_key
    raw_df = cross_run_metric_values(resolved_component_dirs, raw_metric_key).copy()
    raw_df = raw_df[raw_df["arch"].isin(component_included_arches)]
    raw_df["raw_value"] = pd.to_numeric(raw_df["value"], errors="coerce")
    raw_df = raw_df.dropna(subset=["raw_value"])

    raw_min = raw_df["raw_value"].min()
    raw_max = raw_df["raw_value"].max()
    raw_span = raw_max - raw_min

    metric_df = raw_df.copy()
    metric_df["component"] = component
    metric_df["metric"] = metric_key
    metric_df["raw_metric"] = raw_metric_key

    if raw_span == 0:
        metric_df["normalized_value"] = 0.5
        formula = "0.5 because all raw values are equal"
    elif metric_key == "wall_min_distance_risk":
        metric_df["normalized_value"] = (raw_max - metric_df["raw_value"]) / raw_span
        formula = f"({raw_max:.6g} - raw_value) / ({raw_max:.6g} - {raw_min:.6g})"
    else:
        metric_df["normalized_value"] = (metric_df["raw_value"] - raw_min) / raw_span
        formula = f"(raw_value - {raw_min:.6g}) / ({raw_max:.6g} - {raw_min:.6g})"

    component_math_frames.append(metric_df)
    component_normalization_rows.append(
        {
            "component": component,
            "metric": metric_key,
            "raw_metric": raw_metric_key,
            "raw_min": raw_min,
            "raw_max": raw_max,
            "n_values": metric_df["normalized_value"].count(),
            "formula": formula,
        }
    )

component_math_values = pd.concat(component_math_frames, ignore_index=True)
component_normalization_audit = pd.DataFrame(component_normalization_rows)

arch8_math_by_training_track = (
    component_math_values[component_math_values["arch"] == "arch8"]
    .groupby(["run_dir", "component"], as_index=False)
    .agg(
        normalized_mean=("normalized_value", "mean"),
        raw_mean=("raw_value", "mean"),
        n_testing_track_values=("normalized_value", "count"),
    )
)
arch8_math_by_training_track["training_track"] = arch8_math_by_training_track["run_dir"].map(
    lambda run_dir: TRAINED_MAP_LABELS.get(run_dir, run_dir)
)

baseline_math_by_component = (
    component_math_values[component_math_values["arch"].isin(component_baseline_arches)]
    .groupby("component", as_index=False)
    .agg(
        baseline_normalized_mean=("normalized_value", "mean"),
        baseline_raw_mean=("raw_value", "mean"),
        n_baseline_values=("normalized_value", "count"),
    )
)

arch8_math_by_component = (
    component_math_values[component_math_values["arch"] == "arch8"]
    .groupby("component", as_index=False)
    .agg(
        arch8_normalized_mean=("normalized_value", "mean"),
        arch8_raw_mean=("raw_value", "mean"),
        n_arch8_values=("normalized_value", "count"),
    )
)

component_net_improvement_audit = arch8_math_by_component.merge(
    baseline_math_by_component,
    on="component",
    how="outer",
)
component_net_improvement_audit["net_improvement_%"] = 100.0 * (
    component_net_improvement_audit["baseline_normalized_mean"]
    - component_net_improvement_audit["arch8_normalized_mean"]
) / component_net_improvement_audit["baseline_normalized_mean"]
component_net_improvement_audit = component_net_improvement_audit.set_index("component").reindex(component_order)

print("Included training tracks:", ", ".join(TRAINED_MAP_LABELS.get(run_dir.name, run_dir.name) for run_dir in resolved_component_dirs))
print("Included baseline architectures:", component_baseline_label)
print("Normalization formulas. Lower normalized values are better for every component.")
display(
    component_normalization_audit.set_index("component").reindex(component_order).style.format(
        {
            "raw_min": "{:.6g}",
            "raw_max": "{:.6g}",
            "n_values": "{:.0f}",
        }
    )
)

print("A* training-track component means: mean(normalized_value) over testing tracks")
display(
    arch8_math_by_training_track[
        ["training_track", "run_dir", "component", "normalized_mean", "raw_mean", "n_testing_track_values"]
    ].style.format(
        {
            "normalized_mean": "{:.6f}",
            "raw_mean": "{:.6g}",
            "n_testing_track_values": "{:.0f}",
        }
    )
)

print("Net-improvement math by component")
print("net_improvement_% = 100 * (baseline_normalized_mean - arch8_normalized_mean) / baseline_normalized_mean")
display(
    component_net_improvement_audit.style.format(
        {
            "arch8_normalized_mean": "{:.6f}",
            "arch8_raw_mean": "{:.6g}",
            "n_arch8_values": "{:.0f}",
            "baseline_normalized_mean": "{:.6f}",
            "baseline_raw_mean": "{:.6g}",
            "n_baseline_values": "{:.0f}",
            "net_improvement_%": "{:+.3f}%",
        }
    )
)

print("Full long-form audit rows used in the means")
display(
    component_math_values[
        [
            "run_dir",
            "map_label",
            "arch",
            "component",
            "raw_metric",
            "raw_value",
            "normalized_value",
        ]
    ].sort_values(["component", "run_dir", "map_label", "arch"]).style.format(
        {
            "raw_value": "{:.6g}",
            "normalized_value": "{:.6f}",
        }
    )
)
